# 5. MODEL EVALUATION AND DIAGNOSTICS
## Daily Customer Churn Predictor · VivaMarket Brasil

---

**INPUT:** `../models/churn_model_YYYYMMDD.joblib`, `../data/processed/churn_features_YYYYMMDD.parquet`, and `../data/processed/churn_predictions_YYYYMMDD.parquet`

*The selected churn model, the full feature matrix, and the scored test snapshots produced in NB04.*

**OUTPUT:** `../data/processed/churn_diagnostics_YYYYMMDD.csv` and `../reports/model_diagnostics_YYYYMMDD.html`

*A business-facing diagnostic package covering ranking quality, calibration, temporal stability, and threshold trade-offs.*


---
## 5.1. STARTING SITUATION


NB04 selected the best-performing model under a temporal split on the canonical V2C formulation and produced a scored test population with provisional percentile-based risk tiers. Before moving into explainability and deployment, the project needs a more rigorous view of how reliable those scores really are under the still-positive-heavy v2 target.

This notebook therefore turns raw predictive output into a **decision-quality diagnostic layer**. The goal is to verify whether the model remains useful across future monthly snapshots, how concentrated risk is at the top of the ranking, and how calibration and threshold choices affect the retention workload.

---
## 5.2. NOTEBOOK OBJECTIVE


- **Business objective:** verify that the selected churn model prioritizes the right customers for retention actions and supports economically reasonable contact thresholds.
- **Analytical objective:** quantify ranking quality, calibration, temporal stability, and campaign concentration across the held-out test period for the canonical V2C line.

---
## 5.3. INITIAL SETUP

**What is done**

We load the libraries required for diagnostics, plotting, model loading, and HTML report generation.

**Why it is done**

NB05 must be reproducible and explicit because the evaluation stage is where modeling quality becomes a business decision.

**Expected result**

A stable environment with resolved paths, active logging, and report folders ready for diagnostic outputs.


In [1]:
import base64
import io
import logging
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve, roc_auc_score, roc_curve

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', force=True)
logger = logging.getLogger('nb05_model_evaluation')
logger.info('NB05 started: model evaluation and diagnostics.')

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')



2026-05-31 17:18:29,837 | INFO | NB05 started: model evaluation and diagnostics.


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

run_timestamp = datetime.now(ZoneInfo('Europe/Paris'))
run_datetime_label = run_timestamp.strftime('%Y-%m-%d %H:%M %Z')
scoring_package_path = sorted(MODELS_DIR.glob('churn_scoring_package_*.joblib'))[-1]
bundle = joblib.load(scoring_package_path)
model_package = bundle['model_package']
metadata = bundle.get('metadata', {})
run_date_tag = metadata.get('run_date_tag', scoring_package_path.stem.split('_')[-1])
prediction_path = PROCESSED_DIR / 'churn_predictions_20260506.parquet'
feature_path = PROCESSED_DIR / 'churn_features_20260506.parquet'
diagnostics_csv_path = PROCESSED_DIR / f'churn_diagnostics_{run_date_tag}.csv'
diagnostics_html_path = REPORTS_DIR / f'model_diagnostics_{run_date_tag}.html'
calibration_csv_path = PROCESSED_DIR / f'churn_calibration_comparison_{run_date_tag}.csv'

if not feature_path.exists():
    raise FileNotFoundError(f'Expected feature artifact for diagnostics: {feature_path}')
if not prediction_path.exists():
    raise FileNotFoundError(f'Expected prediction artifact for diagnostics: {prediction_path}')

logger.info('Scoring package path: %s', scoring_package_path)
logger.info('Feature path: %s', feature_path)
logger.info('Prediction path: %s', prediction_path)
logger.info('Diagnostics run anchored to run_date_tag=%s', run_date_tag)



2026-05-31 17:18:29,870 | INFO | Scoring package path: /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/models/churn_scoring_package_20260519.joblib


2026-05-31 17:18:29,872 | INFO | Feature path: /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_features_20260506.parquet


2026-05-31 17:18:29,873 | INFO | Prediction path: /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_predictions_20260506.parquet


2026-05-31 17:18:29,874 | INFO | Diagnostics run anchored to run_date_tag=20260519


---
## 5.4. DATA RECONSTRUCTION FOR DIAGNOSTICS

**What is done**

We reload the model package, rebuild the same encoded feature space used in NB04, and align the test rows with the persisted scored output.

**Why it is done**

Deep diagnostics need access to both the full feature matrix and the production-like prediction file so that ranking, stability and threshold analysis remain fully auditable.

**Expected result**

A matched diagnostic table for the held-out test period, including probabilities, labels, snapshot dates and campaign tiers.


In [3]:
feature_df = pd.read_parquet(feature_path)
feature_df['snapshot_date'] = pd.to_datetime(feature_df['snapshot_date'])
prediction_df = pd.read_parquet(prediction_path)
prediction_df['snapshot_date'] = pd.to_datetime(prediction_df['snapshot_date'])

test_keys = model_package['test_snapshot_keys']
validation_keys = model_package['validation_snapshot_keys']
feature_columns = model_package['feature_columns']
scored_model = model_package['model']
target_column = model_package.get('target_column', metadata.get('target_column', 'churn_v2_label' if 'churn_v2_label' in feature_df.columns else 'churn_90d_label'))

leakage_columns = [
    'customer_unique_id', 'snapshot_key', 'snapshot_date', 'first_purchase_timestamp',
    'last_purchase_timestamp', 'future_orders_90d', 'future_revenue_90d', 'churn_90d_label',
    'churn_v2_label', 'next_purchase_timestamp', 'days_to_next_purchase', 'future_purchase_within_horizon'
]
feature_columns_source = [c for c in feature_df.columns if c not in leakage_columns]

def encode_frame(frame: pd.DataFrame) -> pd.DataFrame:
    encoded = pd.get_dummies(frame[feature_columns_source], columns=['customer_state'], dtype=float)
    return encoded.reindex(columns=feature_columns, fill_value=0.0)

validation_df = feature_df[feature_df['snapshot_key'].astype(str).isin([str(k) for k in validation_keys])].copy()
X_validation = encode_frame(validation_df)
y_validation = validation_df[target_column].astype(int)

test_df = feature_df[feature_df['snapshot_key'].astype(str).isin([str(k) for k in test_keys])].copy()
X_test = encode_frame(test_df)
test_df['recomputed_probability'] = scored_model.predict_proba(X_test)[:, 1]
test_df['observed_target'] = test_df[target_column].astype(int)

diagnostics_df = prediction_df.merge(
    test_df[[
        'customer_unique_id', 'snapshot_key', 'snapshot_date', 'recomputed_probability', 'observed_target'
    ]],
    on=['customer_unique_id', 'snapshot_key', 'snapshot_date'],
    how='left',
    validate='one_to_one',
)
diagnostics_df['observed_target'] = diagnostics_df['observed_target_x'].fillna(diagnostics_df['observed_target_y']).astype(int)
diagnostics_df = diagnostics_df.drop(columns=['observed_target_x', 'observed_target_y'])
diagnostics_df['probability_diff'] = diagnostics_df['churn_probability'] - diagnostics_df['recomputed_probability']
logger.info('Target column used for diagnostics: %s', target_column)
logger.info('Maximum scoring reconstruction difference: %.10f', diagnostics_df['probability_diff'].abs().max())
diagnostics_df.head()
model_version = metadata.get('model_version', metadata.get('version_name', 'v2'))
pipeline_tag = metadata.get('pipeline_tag', 'canonical_v2c_phase2')
run_id = metadata.get('run_id', f'canonical_v2c_{run_date_tag}')



2026-05-31 17:18:29,971 | INFO | Target column used for diagnostics: churn_v2_label


2026-05-31 17:18:29,972 | INFO | Maximum scoring reconstruction difference: 0.0000000000


---
## 5.5. GLOBAL PERFORMANCE AND THRESHOLD TRADE-OFFS

**What is done**

We quantify overall ranking quality, threshold trade-offs, and campaign concentration at the top of the score distribution.

**Why it is done**

Retention budgets care about who appears first in the ranking, how many customers would be contacted, and what observed churn rate sits inside each operational slice.

**Expected result**

A concise metric package that links predictive performance to the High / Medium / Low retention framework.


In [4]:
def precision_at_top_fraction(y_true: pd.Series, scores: pd.Series, fraction: float) -> float:
    rank_df = pd.DataFrame({'y_true': y_true.to_numpy(), 'score': scores.to_numpy()})
    rank_df = rank_df.sort_values('score', ascending=False).reset_index(drop=True)
    cutoff = max(int(np.ceil(len(rank_df) * fraction)), 1)
    return float(rank_df.head(cutoff)['y_true'].mean())

y_true = diagnostics_df['observed_target'].astype(int)
y_score = diagnostics_df['churn_probability'].astype(float)

metric_rows = [
    ('roc_auc', roc_auc_score(y_true, y_score)),
    ('average_precision', average_precision_score(y_true, y_score)),
    ('brier_score', brier_score_loss(y_true, y_score)),
    ('label_prevalence', float(y_true.mean())),
    ('precision_at_top_1pct', precision_at_top_fraction(y_true, y_score, 0.01)),
    ('precision_at_top_5pct', precision_at_top_fraction(y_true, y_score, 0.05)),
    ('precision_at_top_10pct', precision_at_top_fraction(y_true, y_score, 0.10)),
    ('high_risk_share', float((diagnostics_df['risk_tier'] == 'HIGH').mean())),
    ('medium_risk_share', float((diagnostics_df['risk_tier'] == 'MEDIUM').mean())),
    ('low_risk_share', float((diagnostics_df['risk_tier'] == 'LOW').mean())),
]
summary_metrics = pd.DataFrame(metric_rows, columns=['metric', 'value'])
summary_metrics

,metric,value
0,roc_auc,0.8016
1,average_precision,0.9937
2,brier_score,0.0572
3,label_prevalence,0.9770
4,precision_at_top_1pct,1.0000
5,precision_at_top_5pct,1.0000
6,precision_at_top_10pct,0.9970
7,high_risk_share,0.2002
8,medium_risk_share,0.2998
9,low_risk_share,0.5000


In [5]:
quantile_grid = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
threshold_rows = []
for quantile in quantile_grid:
    threshold = float(diagnostics_df['churn_probability'].quantile(quantile))
    targeted = diagnostics_df['churn_probability'] >= threshold
    contacts = int(targeted.sum())
    if contacts:
        targeted_frame = diagnostics_df.loc[targeted].copy()
        precision = float(targeted_frame['observed_target'].mean())
        recall = float(targeted_frame['observed_target'].sum() / diagnostics_df['observed_target'].sum())
    else:
        precision = np.nan
        recall = np.nan
    threshold_rows.append({
        'score_quantile_cutoff': quantile,
        'score_threshold': threshold,
        'targeted_rows': contacts,
        'targeted_share': float(targeted.mean()),
        'observed_churn_rate': precision,
        'recall_at_threshold': recall,
        'avg_total_payment_value': float(diagnostics_df.loc[targeted, 'total_payment_value'].mean()) if contacts else np.nan,
    })
threshold_df = pd.DataFrame(threshold_rows)
threshold_df


,score_quantile_cutoff,score_threshold,targeted_rows,targeted_share,observed_churn_rate,recall_at_threshold,avg_total_payment_value
0,0.5000,0.9558,1673,0.5000,0.9958,0.5096,261.9451
1,0.6000,0.9714,1339,0.4002,0.9963,0.4081,245.4430
2,0.7000,0.9808,1004,0.3001,0.9980,0.3065,243.7155
3,0.8000,0.9877,670,0.2002,0.9985,0.2046,251.8046
4,0.9000,0.9929,335,0.1001,0.9970,0.1022,248.0339
5,0.9500,0.9949,169,0.0505,1.0000,0.0517,250.8551


---
## 5.6. TEMPORAL STABILITY AND CALIBRATION

**What is done**

We evaluate diagnostics by monthly snapshot and compare raw versus calibrated probabilities through decile-level reliability checks.

**Why it is done**

A model that looks good in aggregate can still drift month to month or overstate confidence in certain parts of the score range. For a positive-heavy target, it is important to separate **ranking usefulness** from **probability interpretability**.

**Expected result**

A stable monthly view plus an evidence-based conclusion on whether the canonical V2C line should be interpreted mainly as a ranking-first model or as a probability signal that benefits materially from calibration.



In [6]:
monthly_backtest = (
    diagnostics_df.groupby('snapshot_key', observed=True)
    .apply(lambda frame: pd.Series({
        'rows_n': len(frame),
        'observed_churn_rate': frame['observed_target'].mean(),
        'avg_score': frame['churn_probability'].mean(),
        'precision_at_top_10pct': precision_at_top_fraction(frame['observed_target'].astype(int), frame['churn_probability'], 0.10),
        'average_precision': average_precision_score(frame['observed_target'].astype(int), frame['churn_probability']),
    }), include_groups=False)
    .reset_index()
)
monthly_backtest

,snapshot_key,rows_n,observed_churn_rate,avg_score,precision_at_top_10pct,average_precision
0,20180401,"1,551.0000",0.9761,0.8713,0.9936,0.9933
1,20180501,"1,795.0000",0.9777,0.8694,1.0000,0.9941


In [7]:
monthly_backtest = (
    diagnostics_df.groupby('snapshot_key', observed=True)
    .apply(lambda frame: pd.Series({
        'rows_n': len(frame),
        'observed_churn_rate': frame['observed_target'].mean(),
        'avg_score': frame['churn_probability'].mean(),
        'precision_at_top_10pct': precision_at_top_fraction(frame['observed_target'].astype(int), frame['churn_probability'], 0.10),
        'average_precision': average_precision_score(frame['observed_target'].astype(int), frame['churn_probability']),
    }), include_groups=False)
    .reset_index()
)

sigmoid_calibrator = CalibratedClassifierCV(FrozenEstimator(scored_model), method='sigmoid')
sigmoid_calibrator.fit(X_validation, y_validation)
isotonic_calibrator = CalibratedClassifierCV(FrozenEstimator(scored_model), method='isotonic')
isotonic_calibrator.fit(X_validation, y_validation)

diagnostics_df['churn_probability_sigmoid'] = sigmoid_calibrator.predict_proba(X_test)[:, 1]
diagnostics_df['churn_probability_isotonic'] = isotonic_calibrator.predict_proba(X_test)[:, 1]

def build_calibration_table(label: str, probabilities: pd.Series) -> pd.DataFrame:
    bins = pd.qcut(probabilities, q=10, duplicates='drop')
    table = (
        pd.DataFrame({
            'calibration_bin': bins,
            'predicted_probability': probabilities.astype(float),
            'observed_target': y_true,
        })
        .groupby('calibration_bin', observed=True)
        .agg(
            customers=('observed_target', 'size'),
            avg_predicted_probability=('predicted_probability', 'mean'),
            observed_churn_rate=('observed_target', 'mean'),
        )
        .reset_index()
    )
    table['absolute_calibration_gap'] = (table['avg_predicted_probability'] - table['observed_churn_rate']).abs()
    table['model_variant'] = label
    return table

calibration_frames = [
    build_calibration_table('raw', diagnostics_df['churn_probability']),
    build_calibration_table('sigmoid', diagnostics_df['churn_probability_sigmoid']),
    build_calibration_table('isotonic', diagnostics_df['churn_probability_isotonic']),
]
calibration_df = pd.concat(calibration_frames, ignore_index=True)

comparison_rows = []
for label, probabilities in [
    ('raw', diagnostics_df['churn_probability']),
    ('sigmoid', diagnostics_df['churn_probability_sigmoid']),
    ('isotonic', diagnostics_df['churn_probability_isotonic']),
]:
    model_table = calibration_df[calibration_df['model_variant'] == label].copy()
    top_bin_gap = float(model_table.sort_values('avg_predicted_probability').iloc[-1]['absolute_calibration_gap'])
    comparison_rows.append({
        'model_variant': label,
        'roc_auc': roc_auc_score(y_true, probabilities),
        'average_precision': average_precision_score(y_true, probabilities),
        'brier_score': brier_score_loss(y_true, probabilities),
        'mean_calibration_gap': float(model_table['absolute_calibration_gap'].mean()),
        'top_decile_gap': top_bin_gap,
    })

calibration_comparison_df = pd.DataFrame(comparison_rows).sort_values('mean_calibration_gap').reset_index(drop=True)
calibration_csv_path.parent.mkdir(parents=True, exist_ok=True)
calibration_comparison_df.to_csv(calibration_csv_path, index=False)
logger.info('Calibration comparison CSV saved to %s', calibration_csv_path)

best_variant_row = calibration_comparison_df.iloc[0]
raw_row = calibration_comparison_df[calibration_comparison_df['model_variant'] == 'raw'].iloc[0]
if best_variant_row['model_variant'] == 'raw':
    calibration_takeaway = 'Raw probabilities remain the clearest choice for this notebook because calibration did not improve the reliability profile enough to justify replacing the original ranking signal.'
    calibration_conclusion = 'Ranking-first baseline; no calibrated variant earned promotion.'
elif best_variant_row['model_variant'] == 'sigmoid':
    calibration_takeaway = 'Sigmoid calibration improved the reliability profile while preserving ranking performance, making it the clearest interpretation upgrade for the canonical V2C line.'
    calibration_conclusion = 'Ranking-first model with probability interpretation upgraded by sigmoid calibration.'
else:
    calibration_takeaway = 'Isotonic calibration minimized the average reliability gap, but its ranking tradeoff means the canonical V2C line should still be communicated primarily as a ranking-first system.'
    calibration_conclusion = 'Probabilities are more reliable under isotonic, but the system remains ranking-first overall.'

calibration_decision_df = pd.DataFrame([
    {
        'winning_variant': best_variant_row['model_variant'].title(),
        'mean_calibration_gap': best_variant_row['mean_calibration_gap'],
        'ranking_delta_vs_raw': best_variant_row['roc_auc'] - raw_row['roc_auc'],
        'brier_score': best_variant_row['brier_score'],
        'conclusion': calibration_conclusion,
    }
])

calibration_comparison_df




2026-05-31 17:18:30,209 | INFO | Calibration comparison CSV saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_calibration_comparison_20260519.csv


,model_variant,roc_auc,average_precision,brier_score,mean_calibration_gap,top_decile_gap
0,isotonic,0.7919,0.9920,0.0229,0.0082,0.0000
1,sigmoid,0.8016,0.9937,0.0224,0.0109,0.0052
2,raw,0.8016,0.9937,0.0572,0.1066,0.0018


---
## 5.7. DIAGNOSTIC VISUALS AND HTML REPORT


In [8]:
def figure_to_base64(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', bbox_inches='tight', dpi=160)
    plt.close(fig)
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

roc_fpr, roc_tpr, _ = roc_curve(y_true, y_score)
pr_precision, pr_recall, _ = precision_recall_curve(y_true, y_score)
prob_true, prob_pred = calibration_curve(y_true, y_score, n_bins=10, strategy='quantile')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].plot(roc_fpr, roc_tpr, label=f'ROC AUC = {roc_auc_score(y_true, y_score):.3f}', color='#1f77b4')
axes[0].plot([0, 1], [0, 1], linestyle='--', color='grey')
axes[0].set_title('ROC CURVE')
axes[0].legend()

axes[1].plot(prob_pred, prob_true, marker='o', color='#2ca02c')
axes[1].plot([0, 1], [0, 1], linestyle='--', color='grey')
axes[1].set_title('CALIBRATION CURVE')
axes[1].set_xlabel('Predicted probability')
axes[1].set_ylabel('Observed churn rate')
plt.tight_layout()
static_chart = figure_to_base64(fig)

risk_summary = (
    diagnostics_df.groupby('risk_tier', observed=False)
    .agg(
        rows_n=('customer_unique_id', 'size'),
        customers_n=('customer_unique_id', 'nunique'),
        observed_churn_rate=('observed_target', 'mean'),
        avg_probability=('churn_probability', 'mean')
    )
    .reset_index()
)
risk_summary['risk_tier'] = pd.Categorical(risk_summary['risk_tier'], categories=['HIGH', 'MEDIUM', 'LOW'], ordered=True)
risk_summary = risk_summary.sort_values('risk_tier')

risk_donut = px.pie(
    risk_summary,
    names='risk_tier',
    values='rows_n',
    hole=0.55,
    color='risk_tier',
    color_discrete_map={'HIGH': '#c0392b', 'MEDIUM': '#d4a017', 'LOW': '#2e8b57'},
    title='Risk Mix Across Diagnostic Rows'
)
risk_donut.update_traces(textposition='inside', textinfo='percent+label')
risk_donut.update_layout(margin=dict(t=60, b=20, l=20, r=20), legend_title_text='Risk tier')

threshold_scatter = go.Figure()
threshold_scatter.add_trace(go.Scatter(
    x=threshold_df['score_quantile_cutoff'],
    y=threshold_df['observed_churn_rate'],
    mode='lines+markers',
    name='Precision / observed churn rate',
    line=dict(color='#c0392b', width=3),
))
threshold_scatter.add_trace(go.Scatter(
    x=threshold_df['score_quantile_cutoff'],
    y=threshold_df['recall_at_threshold'],
    mode='lines+markers',
    name='Recall',
    line=dict(color='#1f77b4', width=3),
    yaxis='y2',
))
threshold_scatter.update_layout(
    title='Precision / Recall Trade-off by Quantile Cutoff',
    xaxis=dict(title='Score quantile cutoff', tickformat='.0%'),
    yaxis=dict(title='Precision', rangemode='tozero'),
    yaxis2=dict(title='Recall', overlaying='y', side='right', rangemode='tozero'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(t=70, b=40, l=50, r=50),
)

monthly_precision_chart = px.line(
    monthly_backtest,
    x='snapshot_key',
    y='precision_at_top_10pct',
    markers=True,
    title='Monthly Precision at Top 10%'
)
monthly_precision_chart.update_traces(line_color='#7f8c8d')
monthly_precision_chart.update_layout(margin=dict(t=60, b=40, l=40, r=20), xaxis_title='Snapshot key', yaxis_title='Precision')

calibration_curve_chart = go.Figure()
color_map = {'raw': '#005090', 'sigmoid': '#E0B000', 'isotonic': '#C01010'}
for variant in ['raw', 'sigmoid', 'isotonic']:
    frame = calibration_df[calibration_df['model_variant'] == variant].copy()
    calibration_curve_chart.add_trace(go.Scatter(
        x=frame['avg_predicted_probability'],
        y=frame['observed_churn_rate'],
        mode='lines+markers',
        name=variant.title(),
        line=dict(color=color_map[variant], width=3),
    ))
calibration_curve_chart.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Perfect calibration',
    line=dict(color='#7f8c8d', dash='dash'),
))
calibration_curve_chart.update_layout(
    title='Calibration Comparison by Probability Decile',
    xaxis_title='Average predicted probability',
    yaxis_title='Observed churn rate',
    margin=dict(t=70, b=40, l=50, r=30),
)

css = """
<style>
:root {
  --vm-primary: #005090;
  --vm-accent: #f39c12;
  --vm-high: #c0392b;
  --vm-medium: #8A6A00;
  --vm-low: #2e8b57;
  --vm-text: #1f2933;
  --vm-muted: #4A5568;
  --vm-border: #d9e2ec;
  --vm-bg: #f7fafc;
  --vm-card: #ffffff;
}
body {font-family: Arial, sans-serif; margin: 32px; color: var(--vm-text); background: var(--vm-bg);}
h1, h2 {color: var(--vm-primary);}
.grid {display: grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap: 20px; margin: 20px 0 28px;}
.card {background: var(--vm-card); border: 1px solid var(--vm-border); border-radius: 14px; padding: 18px 20px; box-shadow: 0 8px 24px rgba(15, 23, 42, 0.06);}
.card h3 {margin: 0 0 10px; font-size: 0.95rem; color: var(--vm-muted); text-transform: uppercase; letter-spacing: 0.04em;}
.card .value {font-size: 2rem; font-weight: 700; color: var(--vm-text);}
.card .note {margin-top: 8px; color: var(--vm-muted); font-size: 0.92rem;}
.section {background: var(--vm-card); border: 1px solid var(--vm-border); border-radius: 14px; padding: 22px; margin: 18px 0; box-shadow: 0 8px 24px rgba(15, 23, 42, 0.04);}
table {border-collapse: collapse; width: 100%; font-size: 0.95rem;}
th, td {border: 1px solid var(--vm-border); padding: 10px; text-align: left;}
th {background: #edf2f7;}
.footer {margin-top: 28px; color: var(--vm-muted); font-size: 0.9rem; border-top: 1px solid var(--vm-border); padding-top: 16px;}
</style>
"""

summary_cards = [
    ('ROC AUC', summary_metrics.loc[summary_metrics['metric'].eq('roc_auc'), 'value'].iloc[0], 'Held-out ranking quality under the canonical V2C line.'),
    ('Average Precision', summary_metrics.loc[summary_metrics['metric'].eq('average_precision'), 'value'].iloc[0], 'Strong precision remains influenced by the positive-heavy target.'),
    ('Precision@Top 10%', summary_metrics.loc[summary_metrics['metric'].eq('precision_at_top_10pct'), 'value'].iloc[0], 'Operational concentration among the top-contact band.'),
    ('Best calibration gap', calibration_comparison_df['mean_calibration_gap'].iloc[0], f"{calibration_decision_df['winning_variant'].iloc[0]} produced the smallest average decile calibration gap across raw, sigmoid, and isotonic variants."),
]
summary_card_html = ''.join([
    f"<div class='card'><h3>{label}</h3><div class='value'>{value:.4f}</div><div class='note'>{note}</div></div>"
    for label, value, note in summary_cards
])

html_parts = [
    '<html><head><meta charset="utf-8"><title>Model Diagnostics</title></head><body>',
    css,
    '<h1>MODEL DIAGNOSTICS REPORT</h1>',
    '<p><strong>Diagnostic context:</strong> Canonical V2C line with percentile-based provisional risk tiers and publication-quality diagnostic reporting.</p>',
    f"<p><strong>Calibration takeaway:</strong> {calibration_takeaway}</p>",
    f"<div class='grid'>{summary_card_html}</div>",
    "<div class='section'><h2>Interactive risk mix</h2><p>The donut highlights how diagnostic rows are distributed across the analytical V2C tiers.</p>" + risk_donut.to_html(full_html=False, include_plotlyjs='cdn') + '</div>',
    "<div class='section'><h2>Executive summary metrics</h2>" + summary_metrics.to_html(index=False) + "<h3 style='margin-top:18px;'>Calibration decision</h3><p>The row below makes the calibration outcome explicit instead of leaving it only as a caution note.</p>" + calibration_decision_df.to_html(index=False) + '</div>',
    "<div class='section'><h2>Risk-tier summary</h2>" + risk_summary.to_html(index=False) + '</div>',
    "<div class='section'><h2>Quantile threshold trade-offs</h2><p>The table and chart below show how contact volume, precision, and recall move together as campaign selectivity changes.</p>" + threshold_df.to_html(index=False) + threshold_scatter.to_html(full_html=False, include_plotlyjs=False) + '</div>',
    "<div class='section'><h2>Monthly backtest</h2>" + monthly_backtest.to_html(index=False) + monthly_precision_chart.to_html(full_html=False, include_plotlyjs=False) + '</div>',
    "<div class='section'><h2>Calibration comparison</h2><p>This section compares whether raw, sigmoid, or isotonic probabilities behave more credibly once the positive-heavy target is translated into decile-level observed rates.</p>" + calibration_comparison_df.to_html(index=False) + calibration_curve_chart.to_html(full_html=False, include_plotlyjs=False) + '</div>',
    "<div class='section'><h2>Calibration bins</h2>" + calibration_df.to_html(index=False) + '</div>',
    f"<div class='section'><h2>Static diagnostic curves</h2><img src='data:image/png;base64,{static_chart}' style='max-width:1100px; width:100%;'></div>",
    f"<div class='footer'><strong>Run date:</strong> {run_datetime_label} &nbsp;|&nbsp; <strong>Model version:</strong> {model_version} &nbsp;|&nbsp; <strong>Pipeline tag:</strong> {pipeline_tag} &nbsp;|&nbsp; <strong>Run id:</strong> {run_id}</div>",
    '</body></html>'
]
diagnostics_html_path.write_text('\n'.join(html_parts), encoding='utf-8')

diagnostics_export = pd.concat([
    summary_metrics.assign(section='summary'),
    threshold_df.assign(section='thresholds'),
    monthly_backtest.assign(section='monthly_backtest'),
    calibration_df.assign(section='calibration'),
    calibration_comparison_df.assign(section='calibration_compare'),
], ignore_index=True, sort=False)
diagnostics_export.to_csv(diagnostics_csv_path, index=False)
logger.info('Diagnostics CSV saved to %s', diagnostics_csv_path)
logger.info('Diagnostics HTML report saved to %s', diagnostics_html_path)





2026-05-31 17:18:30,730 | INFO | Diagnostics CSV saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_diagnostics_20260519.csv


2026-05-31 17:18:30,731 | INFO | Diagnostics HTML report saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/reports/model_diagnostics_20260519.html


---
## 5.8. NOTEBOOK CLOSURE


---
## 5.8. NOTEBOOK CLOSURE

The diagnostic stage confirms whether the selected ranking is stable enough to drive retention actions. The main operational value of this notebook is that it reframes model quality as **campaign quality**: who gets contacted, how much churn concentrates in the top ranks, and how stable the score remains across future monthly snapshots.

Under the canonical V2C formulation, the diagnostics now separate two questions clearly:

1. **is the ranking strong enough to prioritize customers?**
2. **are the probabilities themselves reliable enough to communicate as business-like risk levels?**

This matters because the target remains highly positive. A strong ROC AUC or precision concentration does not automatically mean that the raw probabilities should be read literally as real-world churn likelihoods.

The calibration comparison therefore becomes part of the formal project evidence: if sigmoid or isotonic materially improves reliability, the portfolio can say so explicitly; if not, the honest conclusion is that the system remains primarily a **ranking-first** decision layer.

The next notebook should explain *why* those customers score high risk by translating the model into SHAP-based churn drivers and segment-level narratives.

